

## 1）Seq2Seq 论文要点（以 Sutskever et al., 2014《Sequence to Sequence Learning with Neural Networks》为主）

### 研究目标

* 把**输入序列**（如英文句子）映射成**输出序列**（如法文句子），长度可变。
* 用一个统一的神经网络端到端学习：
  [
  $$
  p(y_1,\dots,y_T|x_1,\dots,x_S)=\prod_{t=1}^{T} p(y_t|y_{<t},x)
  $$
  ]

### 核心模型：Encoder–Decoder（LSTM）

* **Encoder（编码器）**：读入 (x_1 \to x_S)，用 LSTM 更新隐藏状态
  [
  $$
  h_i = \text{LSTM}(h_{i-1}, x_i)
  $$
  ]
  最终得到一个“句子表示”（论文里主要用最后的隐藏状态/记忆作为压缩表示）。
* **Decoder（解码器）**：以 encoder 的最终状态作为初始状态，逐步生成 (y_t)
  [
  $$
  s_t=\text{LSTM}(s_{t-1}, y_{t-1}),\quad p(y_t)=\text{Softmax}(Ws_t+b)
  $$
  ]

### 训练方式（Teacher Forcing）

* 训练时 decoder 的输入用真实上一词 (y_{t-1})（而不是模型预测），最大化对数似然（交叉熵）。
* 使用反向传播通过时间（BPTT）训练整个网络。

### 关键技巧：把源序列“反过来”

* 把输入句子顺序反转（reverse source sentence）能显著提升效果。
* 直觉：让源句开头的信息更靠近 decoder 生成开头的位置，**缩短长距离依赖路径**，更容易学。

### 推理（Inference）

* 测试时没有真实 (y_{t-1})，用模型自己前一步输出喂回去自回归生成。
* 常用 **Beam Search** 找更优输出序列（而不是贪心）。

### 主要结论/贡献

* 证明了“**一个大 LSTM**”可以在翻译这种复杂任务上端到端工作。
* 反转输入是简单但非常有效的 trick。
* 说明模型容量（层数/隐藏维度）与效果强相关（大模型更好）。

> 备注：后续你学到的注意力机制（Bahdanau 2015）就是为了解决这里“把整句压成一个向量”导致的信息瓶颈问题；但这篇 2014 的主线就是纯 encoder-decoder LSTM。

---




## 2）Transformer 模型结构及计算过程

### 整体架构

* **Encoder-Decoder 结构**：由多个相同的 Encoder 层堆叠 + 多个相同的 Decoder 层堆叠组成
* **输入处理**：源序列经 Embedding + Positional Encoding → Decoder 输入：目标序列经 Embedding + Positional Encoding + Masked Self-Attention

### 核心组件：Multi-Head Attention

#### 计算过程（以 Self-Attention 为例）：
1. **线性变换**：将输入序列 X 分别投影到 Q（Query）、K（Key）、V（Value）空间
   $$
   Q = XW^Q, \quad K = XW^K, \quad V = XW^V
   $$
   其中 W^Q, W^K, W^V 为可学习参数矩阵

2. **注意力分数计算**：Q 与 K 的点积，得到注意力权重
   $$
   \text{Attention}(Q,K,V) = \text{Softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
   $$
   其中 d_k 为 Key 的维度，除以 \sqrt{d_k} 防止梯度消失

3. **多头机制**：将 Q、K、V 分成多个头（head），并行计算多个注意力
   [
   $$
   \text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h)W^O
   $$
   每个 head_i = Attention(Q_i, K_i, V_i)

### Encoder 层结构

#### 每个 Encoder 层的计算流程：
1. **Multi-Head Self-Attention**：输入序列自身做注意力
   $$
   \text{attn_output} = \text{MultiHead}(X, X, X)
   $$

2. **Add & Norm（残差连接 + 层归一化）**
   $$
   \text{sub_layer1} = \text{LayerNorm}(X + \text{attn_output})
   $$

3. **Position-wise Feed Forward Network**
   $$
   \text{FFN}(x) = \max(0, xW_1 + b_1)W_2 + b_2
   $$
   两个线性变换 + ReLU 激活，每个位置独立计算

4. **再次 Add & Norm**
   $$
   \text{encoder_output} = \text{LayerNorm}(\text{sub_layer1} + \text{FFN}(\text{sub_layer1}))
   $$

### Decoder 层结构

#### 每个 Decoder 层的计算流程：
1. **Masked Multi-Head Self-Attention**：目标序列自身注意力，但只能看到之前的位置（因果掩码）
   [
   $$
   \text{masked_attn} = \text{MaskedMultiHead}(Y, Y, Y)
   $$
   生成时防止看到未来信息

2. **Add & Norm**
   $$
   \text{sub_layer1} = \text{LayerNorm}(Y + \text{masked_attn})
   $$

3. **Encoder-Decoder Attention（交叉注意力）**：Query 来自 Decoder，Key/Value 来自 Encoder
   $$
   \text{cross_attn} = \text{MultiHead}(\text{sub_layer1}, \text{encoder_output}, \text{encoder_output})
   $$
   让 Decoder 关注源序列的相关信息

4. **Add & Norm**
   $$
   \text{sub_layer2} = \text{LayerNorm}(\text{sub_layer1} + \text{cross_attn})
   $$

5. **Position-wise Feed Forward Network**（同 Encoder）
   $$
   \text{decoder_output} = \text{LayerNorm}(\text{sub_layer2} + \text{FFN}(\text{sub_layer2}))
   $$

### 位置编码（Positional Encoding）

* **目的**：弥补 Transformer 没有循环/卷积结构，无法感知位置信息的缺陷
* **计算公式**：
  $$
  PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)
  $$
  $$
  PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)
  $$
  其中 pos 为位置，i 为维度索引，使用正弦余弦函数提供相对位置信息

### 训练与推理特点

* **训练**：并行计算所有位置，效率高；使用 Teacher Forcing
* **推理**：自回归生成，每个时间步只能看到之前生成的 token
* **优势**：长距离依赖建模能力强，并行化程度高

---